In [1]:
%cd ../..

/run/media/nazif/2F946E411BA61D49/mirscribe


In [11]:
import pandas as pd
from ast import literal_eval
import json
from scripts.ensembl import *
g37 = import_pyensembl(37)


INFO:pyensembl.sequence_data:Loaded sequence dictionary from /run/media/nazif/2F946E411BA61D49/data/pyensembl/GRCh37/ensembl75/Homo_sapiens.GRCh37.75.cdna.all.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /run/media/nazif/2F946E411BA61D49/data/pyensembl/GRCh37/ensembl75/Homo_sapiens.GRCh37.75.ncrna.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /run/media/nazif/2F946E411BA61D49/data/pyensembl/GRCh37/ensembl75/Homo_sapiens.GRCh37.75.pep.all.fa.gz.pickle


In [3]:
def explode_lists(value):  # sourcery skip: inline-immediately-returned-variable
    if value == "not_found":
        return ["not_found"]
    
    try:
        value_list = literal_eval(value)
        
        return value_list
    
    except (SyntaxError, ValueError):
        return [value]

In [4]:
df2 = pd.read_csv("results/sana/gain_pairs.csv")
df2 = (df2.assign(exploded=df2.ENST.apply(explode_lists))
      .explode("exploded")
      .drop(columns="ENST")
      .rename(columns={"exploded":"ENST"})
    )

df3 = pd.read_csv("results/sana/loss_pairs.csv")
df3 = (df3.assign(exploded=df3.ENST.apply(explode_lists))
      .explode("exploded")
      .drop(columns="ENST")
      .rename(columns={"exploded":"ENST"})
    )

df2["gain"] = 1
df2["loss"] = 0
df3["gain"] = 0
df3["loss"] = 1
df = pd.concat([df2, df3], ignore_index=True)

In [6]:
# Group mutations by 'mirna_accession' and calculate the total gains, losses, and total mutations
mirna_trends = df.groupby('mirna_accession').agg(
    total_gains=pd.NamedAgg(column='gain', aggfunc='sum'),
    total_losses=pd.NamedAgg(column='loss', aggfunc='sum'),
    total_modifications=pd.NamedAgg(column='id', aggfunc='count')
).reset_index().sort_values("total_modifications", ascending=False)


# Group mutations by 'ENST' (mRNA identifier) and calculate the total gains, losses, and total mutations
mRNA_trends = df.groupby('ENST').agg(
    total_gains=pd.NamedAgg(column='gain', aggfunc='sum'),
    total_losses=pd.NamedAgg(column='loss', aggfunc='sum'),
    total_modifications=pd.NamedAgg(column='id', aggfunc='count')
).reset_index().sort_values("total_modifications", ascending=False)



In [9]:
ensts = mRNA_trends[mRNA_trends.ENST != "not_found"].ENST.to_list()
gene_dict = {}
for i in ensts:
    gene_name = g37.gene_name_of_transcript_id(i)
    gene_dict[i] = gene_name
mRNA_trends['gene_name'] = mRNA_trends['ENST'].map(gene_dict)

In [15]:
json_file_path = "data/gsea/HALLMARK_APOPTOSIS.v2023.1.Hs.json"
with open(json_file_path, "r") as json_file:
    data = json.load(json_file)
if "HALLMARK_APOPTOSIS" in data:
    apoptosis = data["HALLMARK_APOPTOSIS"].get("geneSymbols", [])
else:
    apoptosis = []
# gene_symbols now contains the list of gene symbols

mRNA_trends['in_apoptosis'] = mRNA_trends['gene_name'].apply(lambda x: 1 if x in apoptosis else 0)


In [13]:
import json
json_file_path = "data/gsea/HALLMARK_DNA_REPAIR.v2023.1.Hs.json"
with open(json_file_path, "r") as json_file:
    data = json.load(json_file)
if "HALLMARK_DNA_REPAIR" in data:
    dna_repair = data["HALLMARK_DNA_REPAIR"].get("geneSymbols", [])
else:
    dna_repair = []
# gene_symbols now contains the list of gene symbols

mRNA_trends['in_dna_repair'] = mRNA_trends['gene_name'].apply(lambda x: 1 if x in dna_repair else 0)


In [16]:
mRNA_trends

,ENST,total_gains,total_losses,total_modifications,gene_name,in_dna_repair,in_apoptosis
2896,not_found,30537,31655,62192,NaN,0,0
151,ENST00000269305,4311,6775,11086,TP53,1,0
1293,ENST00000445888,4173,5906,10079,TP53,1,0
1051,ENST00000420246,4173,5897,10070,TP53,1,0
1363,ENST00000455263,4173,5897,10070,TP53,1,0
...,...,...,...,...,...,...,...
350,ENST00000334602,2,1,3,TERT,0,0
149,ENST00000268957,2,1,3,TOB1,0,0
639,ENST00000375459,1,2,3,PCID2,0,0
1701,ENST00000490709,0,1,1,GLIS3,0,0
